# Choosing a Strategy

A generation method should match the shape of the task. Some methods are single-shot judgments: classify this, extract that, return a typed object. Other methods need to look things up, run Python, call helper methods, and refine an answer across multiple steps.

This notebook keeps those two shapes side by side on one `BookshopAgent`:

- `read_the_customer` uses `PredictStrategy` because it turns one line of text into a structured label.
- `recommend_from_shelf` uses the default `CodeActStrategy` because it has to inspect state and use helper methods.

The goal is not to memorize strategy names. The goal is to learn the decision rule: choose the simplest execution shape that can actually solve the method's job.


## Prerequisites

Install NOOA from GitHub with [uv](https://docs.astral.sh/uv/):

```bash
uv add "nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main"
```

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, replace `"your-api-key"` with a real key. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) - those need no API key, just an `api_base`.


## Setup

NOOA works with any LiteLLM-supported model - hosted or local. Pick one below before running the rest of the notebook.


In [ ]:

from nooa.unifiedllm.registry import get_llm_client

# NOOA works with any LiteLLM-supported model - hosted or local.
# Pick one below. Replace "your-api-key" with a real key for hosted providers;
# local providers (Ollama, vLLM) do not need a key, just pass api_base.

model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")                                        # Anthropic
# model = get_llm_client("gpt-5-mini", api_key="your-api-key")                                              # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")                       # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1")                # vLLM (local, no key)
# model = get_llm_client("openai/nvidia/openai/gpt-oss-20b", api_key="your-api-key", api_base="https://inference-api.nvidia.com/v1")  # NVIDIA hosted (OpenAI-compatible)


## The Bookshop Setup

We'll use a small shelf first. Notebook 3 scales the same idea up to the full Project Gutenberg catalog; here the point is just to make the two reasoning shapes easy to compare.


In [ ]:

from typing import Literal

from pydantic import BaseModel, Field

from nooa import Agent, print_prompt, strategy
from nooa.strategies import PredictStrategy


class Mood(BaseModel):
    vibe: Literal["curious", "lost", "hostile", "returning_a_book", "just_browsing"]
    confidence: float = Field(ge=0, le=1)
    note: str = Field(description="A brief private note from the bookseller.")


class Recommendation(BaseModel):
    title: str
    author: str
    reason: str


SHELF = [
    {
        "title": "The Moonstone",
        "author": "Wilkie Collins",
        "language": "en",
        "tags": ["mystery", "classic", "detective"],
        "tone": "patient and clever",
    },
    {
        "title": "Twenty Thousand Leagues under the Seas",
        "author": "Jules Verne",
        "language": "en",
        "tags": ["adventure", "sea", "science"],
        "tone": "curious and expansive",
    },
    {
        "title": "The Yellow Wallpaper",
        "author": "Charlotte Perkins Gilman",
        "language": "en",
        "tags": ["short", "psychological", "classic"],
        "tone": "sharp and unsettling",
    },
    {
        "title": "Candide",
        "author": "Voltaire",
        "language": "en",
        "tags": ["satire", "philosophy", "short"],
        "tone": "acerbic and brisk",
    },
    {
        "title": "The Wind in the Willows",
        "author": "Kenneth Grahame",
        "language": "en",
        "tags": ["comfort", "nature", "friendship"],
        "tone": "gentle and restorative",
    },
]


## Predict: Single-Shot Structured Judgment

A customer's opening line is small, self-contained input. The method does not need to inspect a database or run calculations; it just needs to return a valid `Mood`. That is exactly the shape `PredictStrategy` is for.


In [ ]:

class BookshopAgent(Agent, llm=model):
    """You are a perceptive bookseller in a used bookshop."""

    @strategy(PredictStrategy())
    async def read_the_customer(self, opening_line: str) -> Mood:
        """Classify the customer's vibe from their opening line.

        Pick the single best-fitting vibe and include a concise private note.
        """
        ...


agent = BookshopAgent()

openers = [
    "hi, do you have any Murakami?",
    "this book fell apart in my bag",
    "just looking, thanks",
    "your website said you'd have this in stock",
]

for line in openers:
    mood = await agent.read_the_customer(line)
    print(f"Customer: {line!r}")
    print(f"  vibe: {mood.vibe}  confidence={mood.confidence:.2f}")
    print(f"  note: {mood.note}")
    print()


The output is a real `Mood` object. The return type becomes a schema, and invalid output is retried inside the framework before your Python caller sees it.


In [ ]:

await print_prompt(agent.read_the_customer, opening_line="just looking, thanks")


What to notice in the prompt: there is no REPL and no tool loop. The model sees the task, the rendered argument value, and the schema it must satisfy. One LLM call goes in; one validated object comes out.


## CodeAct: When The Method Has To Do Work

Now change the task. A customer asks for a recommendation, and the answer depends on what the shop actually has. The method needs to look through shelf data, maybe try a couple of searches, then choose. That is no longer a pure classification problem.

CodeAct is the default strategy, so the method below has no `@strategy(...)` decorator. The LLM gets a Python REPL and can call regular methods on `self`.


In [ ]:

class BookshopAgent(Agent, llm=model):
    """You are a perceptive bookseller in a used bookshop."""

    def __init__(self, shelf: list[dict]):
        super().__init__()
        self.shelf = shelf

    def find_books(self, query: str, n: int = 5) -> list[dict]:
        """Return up to n shelf books matching words in the query."""
        terms = [term for term in query.lower().replace(",", " ").split() if len(term) > 2]
        matches = []
        for book in self.shelf:
            haystack = " ".join(
                [book["title"], book["author"], book["language"], book["tone"], *book["tags"]]
            ).lower()
            if any(term in haystack for term in terms):
                matches.append(book)
        return matches[:n]

    @strategy(PredictStrategy())
    async def read_the_customer(self, opening_line: str) -> Mood:
        """Classify the customer's vibe from their opening line."""
        ...

    async def recommend_from_shelf(self, customer_wants: str) -> Recommendation:
        """Recommend one book from the shelf.

        Use self.find_books to inspect plausible candidates. Pick a book that
        fits the request and explain the choice in one or two sentences.
        """
        ...


agent = BookshopAgent(SHELF)
rec = await agent.recommend_from_shelf("something short, sharp, and philosophical")
print(f"Recommended: {rec.title} by {rec.author}")
print(f"Why: {rec.reason}")


## Watch The Two Shapes

Start the trace viewer in a separate terminal and keep it open while you run the next cell:

```bash
nooa start-dev
```

Open [http://localhost:5001](http://localhost:5001). The `read_the_customer` call should look compact. The `recommend_from_shelf` call should show CodeAct's generated Python cells, helper calls, and final `return_result` step.


In [ ]:

mood = await agent.read_the_customer("I want something clever, but please do not make it homework.")
rec = await agent.recommend_from_shelf("something clever but not too long")

print(mood)
print(rec)


This is the main contrast:

- Predict is a bounded model call that returns structured data.
- CodeAct is an iterative Python session that can use state and methods.

The trace viewer makes that difference concrete, which is why it is useful to introduce early.


## Strategies Compose

A `PredictStrategy` method is still just a method on `self`. A CodeAct method can call it as one step in a larger workflow.


In [ ]:

class BookshopAgent(Agent, llm=model):
    """You are a perceptive bookseller in a used bookshop."""

    def __init__(self, shelf: list[dict]):
        super().__init__()
        self.shelf = shelf

    def find_books(self, query: str, n: int = 5) -> list[dict]:
        """Return up to n shelf books matching words in the query."""
        terms = [term for term in query.lower().replace(",", " ").split() if len(term) > 2]
        matches = []
        for book in self.shelf:
            haystack = " ".join(
                [book["title"], book["author"], book["language"], book["tone"], *book["tags"]]
            ).lower()
            if any(term in haystack for term in terms):
                matches.append(book)
        return matches[:n]

    @strategy(PredictStrategy())
    async def read_the_customer(self, opening_line: str) -> Mood:
        """Classify the customer's vibe from their opening line."""
        ...

    async def serve_customer(self, opening_line: str) -> Recommendation:
        """Read the customer's mood, then recommend one book from the shelf.

        Start by awaiting self.read_the_customer. Then use self.find_books and
        the shelf data to choose a book whose tone fits the mood.
        """
        ...


agent = BookshopAgent(SHELF)
rec = await agent.serve_customer("I suppose I need something clever, unless all you have is nonsense.")
print(f"Recommended: {rec.title} by {rec.author}")
print(f"Why: {rec.reason}")


## When To Use Which Strategy

| Method shape | Use | Why |
|---|---|---|
| Classification, extraction, routing, simple structured judgment | `PredictStrategy` | One call, schema-constrained output, no Python execution |
| Search, computation, helper calls, external tools, multi-step reasoning | `CodeActStrategy` | Persistent REPL, method calls, iteration, structured return validation |
| A workflow with both shapes | Mix them on one class | Strategy is per method, not per agent |

If the method can answer directly from its arguments, start with Predict. If it needs to inspect or act on live Python state, use CodeAct.


## Recap

- Strategy is a per-method execution choice.
- `PredictStrategy` is best for single-shot structured outputs.
- `CodeActStrategy` is best when the method needs Python, tools, state, or iteration.
- A CodeAct method can call a Predict method as a normal method on `self`.
- The trace viewer is the clearest way to see the difference between the two execution shapes.

Notebook 3 keeps the CodeAct side and asks a more practical engineering question: what Python surface should the model see when the live state is large?


## Exercises

1. Add a `PredictStrategy` method `classify_request(customer_wants: str)` that returns `Literal["specific_title", "topic", "mood", "complaint"]`.
2. Add a deterministic helper `books_with_tag(tag: str) -> list[dict]` and update `recommend_from_shelf` to prefer it for topic requests.
3. Change `serve_customer` so hostile customers get the shortest matching recommendation.
4. Open the trace viewer and compare the number of LLM/tool/code steps for `read_the_customer`, `recommend_from_shelf`, and `serve_customer`.
